# Notebook 01 — EDA: taxonomia das violações

Esta seção atualiza o EDA do subconjunto relacionado ao tráfico de pessoas. A variável `violacao` é decomposta em **classe**, **subclasse**, **espécie** e **subespécie**; a abrangência é separada somente quando o sufixo é `NACIONAL` ou `INTERNACIONAL`.

- **Unidade observacional neste notebook:** ocorrência de violação.
- **Unidade analítica no Notebook 02:** denúncia, identificada por `hash`.
- Agregação, one-hot encoding, suporte mínimo e clustering ficam para o Notebook 02.

## 1. Configuração

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

COL_VIOLACAO = "violacao"
COL_DENUNCIA = "hash"

assert "df" in globals(), "Carregue o DataFrame `df` antes desta seção."
assert COL_VIOLACAO in df.columns, f"Coluna ausente: {COL_VIOLACAO}"

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]:,}")
print(f"Hash disponível: {COL_DENUNCIA in df.columns}")

## 2. Compreensão da taxonomia

Hierarquia esperada:

1. classe;
2. subclasse;
3. espécie;
4. subespécie.

Níveis posteriores ao quarto serão preservados como `violacao_nivel_extra_n`. O uso de subclasse ou classe para preencher uma espécie ausente será rastreado em `violacao_especie_origem`.

In [ ]:
display(
    df[COL_VIOLACAO]
    .dropna()
    .astype("string")
    .drop_duplicates()
    .head(20)
    .to_frame()
)

## 3. Qualidade dos dados antes da expansão

In [ ]:
def normalize_taxonomy_source(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s*>\s*", ">", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

violacao_fonte = df[COL_VIOLACAO].astype("string")
violacao_limpa = normalize_taxonomy_source(violacao_fonte)
quantidade_niveis = violacao_limpa.str.count(">").add(1).where(violacao_limpa.notna())

qualidade_taxonomia = pd.Series({
    "total_linhas": len(df),
    "valores_ausentes": int(violacao_limpa.isna().sum()),
    "valores_unicos": int(violacao_limpa.nunique()),
    "espacos_nas_extremidades": int((violacao_fonte.notna() & violacao_fonte.ne(violacao_fonte.str.strip())).sum()),
    "espacos_repetidos": int(violacao_fonte.str.contains(r"\s{2,}", na=False).sum()),
    "linhas_com_mais_de_4_niveis": int(quantidade_niveis.gt(4).sum()),
    "maximo_de_niveis": int(quantidade_niveis.max()) if quantidade_niveis.notna().any() else 0,
})
display(qualidade_taxonomia.to_frame("resultado"))

In [ ]:
mask_nivel_vazio = violacao_limpa.str.contains(r"(^>|>$|>{2,})", regex=True, na=False)
casos_nivel_vazio = df.loc[mask_nivel_vazio, [COL_VIOLACAO]].drop_duplicates()
print(f"Valores distintos com nível vazio: {len(casos_nivel_vazio):,}")
display(casos_nivel_vazio.head(30))

In [ ]:
casos_niveis_extras = (
    pd.DataFrame({
        COL_VIOLACAO: df[COL_VIOLACAO],
        "violacao_fonte_normalizada": violacao_limpa,
        "quantidade_niveis": quantidade_niveis,
    })
    .loc[quantidade_niveis.gt(4)]
    .drop_duplicates()
    .sort_values("quantidade_niveis", ascending=False)
)
print(f"Valores distintos com mais de quatro níveis: {len(casos_niveis_extras):,}")
display(casos_niveis_extras.head(30))

## 4. Feature engineering: expansão hierárquica

A expressão regular retira somente os sufixos reconhecidos `- NACIONAL` e `- INTERNACIONAL`. Outros hífens permanecem no nome da categoria.

In [ ]:
def expand_hierarchical_taxonomy(column: pd.Series) -> pd.DataFrame:
    source = normalize_taxonomy_source(column)
    levels = source.str.split(">", expand=True)

    base_names = [
        "violacao_classe",
        "violacao_subclasse",
        "violacao_especie",
        "violacao_subespecie",
    ]
    extras = max(0, levels.shape[1] - len(base_names))
    names = base_names[:levels.shape[1]] + [
        f"violacao_nivel_extra_{i}" for i in range(1, extras + 1)
    ]
    levels.columns = names

    for name in base_names:
        if name not in levels.columns:
            levels[name] = pd.NA

    for name in levels.columns:
        levels[name] = levels[name].astype("string").str.strip().replace("", pd.NA)

    especie_original = levels["violacao_especie"].copy()
    levels["violacao_especie"] = (
        levels["violacao_especie"]
        .fillna(levels["violacao_subclasse"])
        .fillna(levels["violacao_classe"])
    )

    origem = pd.Series(pd.NA, index=levels.index, dtype="string")
    origem.loc[especie_original.notna()] = "ESPECIE"
    origem.loc[especie_original.isna() & levels["violacao_subclasse"].notna()] = "SUBCLASSE"
    origem.loc[
        especie_original.isna()
        & levels["violacao_subclasse"].isna()
        & levels["violacao_classe"].notna()
    ] = "CLASSE"
    levels["violacao_especie_origem"] = origem

    coverage = levels["violacao_especie"].str.extract(
        r"^(?P<violacao_especie_norm>.*?)"
        r"(?:\s*-\s*(?P<violacao_abrangencia_norm>NACIONAL|INTERNACIONAL))?$"
    )
    coverage["violacao_especie_norm"] = coverage["violacao_especie_norm"].str.strip().replace("", pd.NA)

    return pd.concat([levels, coverage], axis=1)

In [ ]:
taxonomy_features = expand_hierarchical_taxonomy(df[COL_VIOLACAO])
overlap = df.columns.intersection(taxonomy_features.columns)

if len(overlap):
    warnings.warn("Colunas taxonômicas substituídas: " + ", ".join(overlap))
    df_base = df.drop(columns=list(overlap)).copy()
else:
    df_base = df.copy()

df_norm = pd.concat([df_base, taxonomy_features], axis=1)

cols_preview = [
    COL_VIOLACAO, "violacao_classe", "violacao_subclasse",
    "violacao_especie", "violacao_subespecie",
    "violacao_especie_norm", "violacao_abrangencia_norm",
    "violacao_especie_origem",
]
print(f"Dimensão: {df.shape} -> {df_norm.shape}")
display(df_norm[cols_preview].head(20))

## 5. Validação da transformação

In [ ]:
cardinalidade = pd.Series({
    "classes": df_norm["violacao_classe"].nunique(),
    "subclasses": df_norm["violacao_subclasse"].nunique(),
    "especies": df_norm["violacao_especie"].nunique(),
    "especies_normalizadas": df_norm["violacao_especie_norm"].nunique(),
    "subespecies": df_norm["violacao_subespecie"].nunique(),
    "abrangencias": df_norm["violacao_abrangencia_norm"].nunique(),
})
display(cardinalidade.to_frame("n_unique"))

In [ ]:
def frequency_table(series: pd.Series) -> pd.DataFrame:
    result = series.value_counts(dropna=False).rename_axis("categoria").to_frame("frequencia")
    result["percentual"] = 100 * result["frequencia"] / len(series)
    return result

display(frequency_table(df_norm["violacao_especie_origem"]))
display(frequency_table(df_norm["violacao_abrangencia_norm"]))

In [ ]:
mask_hifen_sem_abrangencia = (
    df_norm["violacao_especie"].str.contains("-", regex=False, na=False)
    & df_norm["violacao_abrangencia_norm"].isna()
)
hifen_sem_abrangencia = (
    df_norm.loc[mask_hifen_sem_abrangencia,
                ["violacao_especie", "violacao_especie_norm"]]
    .drop_duplicates()
    .sort_values("violacao_especie")
)
print(f"Espécies com hífen e sem abrangência reconhecida: {len(hifen_sem_abrangencia):,}")
display(hifen_sem_abrangencia.head(30))

In [ ]:
falhas_expansao = df_norm.loc[
    df_norm[COL_VIOLACAO].notna() & df_norm["violacao_especie_norm"].isna(),
    [COL_VIOLACAO, "violacao_classe", "violacao_subclasse", "violacao_especie"],
]
print(f"Linhas não ausentes sem espécie normalizada: {len(falhas_expansao):,}")
display(falhas_expansao.head(30))

## 6. Análise univariada dos níveis taxonômicos

As frequências representam ocorrências, e não suporte entre denúncias.

In [ ]:
def frequency_summary(data: pd.DataFrame, column: str):
    frequency = data[column].value_counts()
    return frequency, frequency.describe()

freq_classe, desc_classe = frequency_summary(df_norm, "violacao_classe")
freq_subclasse, desc_subclasse = frequency_summary(df_norm, "violacao_subclasse")
freq_especie, desc_especie = frequency_summary(df_norm, "violacao_especie_norm")

resumo = pd.concat({
    "classe": desc_classe,
    "subclasse": desc_subclasse,
    "especie_norm": desc_especie,
}, axis=1)
display(resumo)

In [ ]:
print("Classes mais frequentes")
display(freq_classe.head(15).to_frame("frequencia"))
print("Subclasses mais frequentes")
display(freq_subclasse.head(15).to_frame("frequencia"))
print("Espécies mais frequentes")
display(freq_especie.head(20).to_frame("frequencia"))

In [ ]:
print("Subclasses menos frequentes")
display(freq_subclasse.sort_values().head(15).to_frame("frequencia"))
print("Espécies menos frequentes")
display(freq_especie.sort_values().head(20).to_frame("frequencia"))

In [ ]:
def plot_rank_frequency(frequency: pd.Series, title: str):
    ordered = frequency.sort_values(ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(np.arange(1, len(ordered) + 1), ordered.values, marker="o")
    ax.set_yscale("log")
    ax.set_xlabel("Posição no ranking")
    ax.set_ylabel("Frequência de ocorrências (escala log)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_rank_frequency(freq_subclasse, "Distribuição das subclasses")
plot_rank_frequency(freq_especie, "Distribuição das espécies normalizadas")

In [ ]:
def plot_top_categories(frequency: pd.Series, title: str, top_n: int):
    data_plot = frequency.head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(data_plot.index.astype(str), data_plot.values)
    ax.set_xlabel("Frequência de ocorrências")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_top_categories(freq_subclasse, "Subclasses mais frequentes", 15)
plot_top_categories(freq_especie, "Espécies mais frequentes", 20)

## 7. Raridade e concentração

Os limiares são diagnósticos e não critérios automáticos de exclusão. A seleção de atributos ocorrerá no Notebook 02, após o cálculo do suporte por denúncia.

In [ ]:
def rarity_report(frequency: pd.Series, thresholds=(1, 5, 10, 30)):
    rows = []
    for threshold in thresholds:
        rare = frequency[frequency <= threshold]
        rows.append({
            "frequencia_maxima": threshold,
            "categorias": len(rare),
            "percentual_categorias": 100 * len(rare) / len(frequency),
            "ocorrencias": int(rare.sum()),
            "percentual_ocorrencias": 100 * rare.sum() / frequency.sum(),
        })
    return pd.DataFrame(rows)

print("Subclasses")
display(rarity_report(freq_subclasse))
print("Espécies")
display(rarity_report(freq_especie))

In [ ]:
def concentration_report(frequency: pd.Series, shares=(0.50, 0.80, 0.90)):
    ordered = frequency.sort_values(ascending=False)
    cumulative = ordered.cumsum() / ordered.sum()
    rows = []
    for share in shares:
        n = int(np.searchsorted(cumulative.to_numpy(), share, side="left") + 1)
        rows.append({
            "parcela_ocorrencias": share,
            "categorias_necessarias": n,
            "percentual_categorias": 100 * n / len(ordered),
        })
    return pd.DataFrame(rows)

print("Subclasses")
display(concentration_report(freq_subclasse))
print("Espécies")
display(concentration_report(freq_especie))

## 8. Descrição por denúncia, sem criar a matriz de modelagem

Esta célula descreve quantas ocorrências e categorias aparecem por denúncia. A matriz binária denúncia × categoria fica para o Notebook 02.

In [ ]:
if COL_DENUNCIA in df_norm.columns:
    resumo_denuncia = (
        df_norm.groupby(COL_DENUNCIA, dropna=False)
        .agg(
            quantidade_ocorrencias=(COL_VIOLACAO, "size"),
            quantidade_classes=("violacao_classe", "nunique"),
            quantidade_subclasses=("violacao_subclasse", "nunique"),
            quantidade_especies=("violacao_especie_norm", "nunique"),
        )
    )
    display(resumo_denuncia.describe().T)
else:
    print(f"Coluna `{COL_DENUNCIA}` ausente; análise por denúncia não executada.")

## 9. Relatório consolidado da transformação

In [ ]:
relatorio_taxonomia = pd.DataFrame({
    "metrica": [
        "linhas", "violacao_ausente", "classes_distintas",
        "subclasses_distintas", "especies_normalizadas_distintas",
        "subespecies_distintas", "linhas_com_abrangencia",
        "fallback_subclasse", "fallback_classe", "nivel_vazio",
        "mais_de_4_niveis", "falhas_expansao",
    ],
    "valor": [
        len(df_norm), int(df_norm[COL_VIOLACAO].isna().sum()),
        int(df_norm["violacao_classe"].nunique()),
        int(df_norm["violacao_subclasse"].nunique()),
        int(df_norm["violacao_especie_norm"].nunique()),
        int(df_norm["violacao_subespecie"].nunique()),
        int(df_norm["violacao_abrangencia_norm"].notna().sum()),
        int(df_norm["violacao_especie_origem"].eq("SUBCLASSE").sum()),
        int(df_norm["violacao_especie_origem"].eq("CLASSE").sum()),
        int(mask_nivel_vazio.sum()), int(quantidade_niveis.gt(4).sum()),
        len(falhas_expansao),
    ],
})
display(relatorio_taxonomia)

## 10. Conclusão e transição para o Notebook 02

A taxonomia foi expandida com rastreabilidade da origem da espécie e separação controlada da abrangência. A concentração e a raridade observadas neste notebook descrevem linhas de ocorrência e não justificam, isoladamente, exclusão de categorias.

No Notebook 02 serão executadas:

1. seleção das denúncias contendo tráfico de pessoas;
2. recuperação das violações concomitantes;
3. agregação por `hash`;
4. matrizes denúncia × classe, subclasse e espécie;
5. apoio mínimo por denúncia e avaliação de variáveis quase constantes;
6. redução de dimensionalidade e clusterização.

## 11. Exportação opcional

In [ ]:
# Requer pyarrow ou fastparquet para Parquet.
# df_norm.to_parquet("dados_disque100_taxonomia_expandida.parquet", index=False)
# relatorio_taxonomia.to_csv("relatorio_qualidade_taxonomia.csv", index=False, encoding="utf-8-sig")
print("DataFrame preparado: df_norm")